In [3]:
# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

# Pick 9 patients with different complexity scores

In [4]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [5]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


# Load `chunks_df`

In [6]:
# look at all oncology chunks in chunks_df:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_ids = selected_patient_ids

conn = sqlite3.connect(db_path)

# Multiple patient_ids: build an IN (...) placeholder list.
if not patient_ids:
    chunks_df = pd.DataFrame()  # avoid invalid SQL: IN ()
else:
    placeholders = ",".join(["?"] * len(patient_ids))
    query = f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    """
    chunks_df = pd.read_sql_query(query, conn, params=patient_ids)

conn.close()

display(chunks_df.head())
print("Number of all chunks:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
1,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,cd9bf67f542dee2c5c6eb4e889086745d239ff7b,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
2,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,c6f4af530450ec4d38f7f478693b4d62c2b3467c,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
3,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,9d5bdbca525ce28a75b1ad7deff73853cc1b20eb,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
4,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,250fb82c086a015ec8fe85d5feeb46806e632e86,0,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of all chunks: 14390


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
21,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,34e083fbe248b34ab7ff75e989fc91da40a6b511,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
931,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,6c44d78bfe33c70133759592f49f88f9cd26734c,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
932,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,4f181909c2fad14ebbdc20f28f1af14d5dfb89f9,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
933,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,b02dbcb96bafeb63dc73b3c40a9f184cb0e781bf,1,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of oncology chunks: 890


# Test questions

In [8]:
import rag_service as rag

In [13]:
# Helper function to get the answer and judge it
import json
import pandas as pd

def run_judge(question: str, rag_output: dict, search_type: str) -> dict:
    """Judge one answer against exactly the context used to produce it."""

    return rag.evaluate_relevance(
        question=question,
        answer=rag_output["answer"],
        context=rag_output["context"],
        search_type=search_type,
        model="gpt-5.4-mini",
    )


def judge_fields(judge_output: dict) -> dict:
    """Extract the normalized evaluation fields for a table."""

    evaluation = judge_output["evaluation"]

    return {
        "relevance_score": evaluation.get("relevance_score"),
        "groundedness_score": evaluation.get("groundedness_score"),
        "overall_score": evaluation.get("overall_score"),
        "relevance_label": evaluation.get("relevance_label"),
        "groundedness_label": evaluation.get("groundedness_label"),
        "explanation": evaluation.get("explanation"),
    }


def print_comparison_result(
    title: str,
    question_type: str | None,
    output: dict,
    judge: dict,
) -> None:
    """Print the answer and its judge result in a readable notebook format."""

    # evaluation = judge["evaluation"]
    evaluation = judge.get("evaluation", judge)


    print(f"\n{'=' * 100}")
    print(title)
    print(f"Question type: {question_type or 'None — generic baseline'}")
    print("=" * 100)

    print("\n--- Answer ---")
    print(output["answer"])

    print("\n--- Judge evaluation ---")
    print(json.dumps(evaluation, indent=2))

    print("\n--- Generation usage ---")
    print(
        f"Input tokens: {output.get('input_tokens', 0):,} | "
        f"Output tokens: {output.get('output_tokens', 0):,} | "
        f"Cost: ${output.get('total_cost', 0.0):.6f}"
    )

    print("\n--- Judge usage ---")
    print(
        f"Input tokens: {judge['token_stats'].get('input_tokens', 0):,} | "
        f"Output tokens: {judge['token_stats'].get('output_tokens', 0):,} | "
        f"Cost: ${judge['cost'].get('total_cost', 0.0):.6f}"
    )

    print("\n--- Context size ---")
    print(f"{len(output['context']):,} characters")

In [27]:
comparison_question = "Summarize the patient's active and historical conditions, including important chronic diagnoses."
comparison_question2 = "What medications is this patient currently or recently taking?"
comparison_question3 = "Give a concise overview of this patient’s medical background and current care context."

search_type = "hybrid"
model = "gpt-5.4-mini"
num_results = 5

# A high complexity patient

In [10]:
patient_id = "f203e11d-5573-1624-69b8-af8436987b3e" #high complexity patient

# overview_question = "Provide a brief overview of this patient's medical background and current status."
# conditions_question = "What are this patient's main diagnosed conditions and their status?"
# medications_question = "What medications is this patient currently or recently taking?"
# oncology_question = "Summarize this patient's oncology history."

## Without question routing: `question_type=None`

We get a neutral baseline because `prompt_mode` defaults to "summary" and retrieval covers over all document types.

Comparison question 1:
The judge is happy (relevance_score is 2, groundedness_score: 1, overall_score: 1.5, RELEVANT, PARTLY_GROUNDED), but the answer completely misses the cancer and other active diagnosis!

(This is because patient_overview.md contains top 10 diagnostic reports, as an approximation for "recent results", which for complex patients with lots of records is an incomplete source for the summary. Reaching directly to the full record, however, the LLM selects the most serious diagnoses from the life-time record and is not able to infer which are the recent and/or active diagnoses.)

In [16]:
unrouted_out = rag.rag_new(
    query=comparison_question,
    patient_id=patient_id,
    question_type=None,
    num_results=num_results,
    model=model,
    search_type=search_type,
)

unrouted_judge = run_judge(
    question=comparison_question,
    rag_output=unrouted_out,
    search_type=search_type,
)

In [17]:
print_comparison_result(
    title="WITHOUT QUESTION ROUTING",
    question_type=None,
    output=unrouted_out,
    judge=unrouted_judge,
)


WITHOUT QUESTION ROUTING
Question type: None — generic baseline

--- Answer ---
1. **Summary:**
- The documented conditions in this record include active stress and historical reports of violence in the environment, with the latter documented as resolved.
- The most recent clinically important status is an active stress finding recorded on 2022-02-15, while the reports of violence in the environment finding was resolved by 2019-07-16.
- No oncology timeline information documented.

2. **Active Conditions:**
- **Stress (finding)** — active; date: 2022-02-15

3. **Medications:**
No current medication is documented in the provided medication snapshot.

4. **Oncology timeline:**
No oncology timeline information documented.

--- Judge evaluation ---
{
  "relevance_score": 2,
  "groundedness_score": 1,
  "overall_score": 1.5,
  "relevance_label": "RELEVANT",
  "groundedness_label": "PARTLY_GROUNDED",
  "explanation": "The answer directly summarizes active and historical conditions, which ma

In [28]:
unrouted_out = rag.rag_new(
    query=comparison_question3,
    patient_id=patient_id,
    question_type=None,
    num_results=num_results,
    model=model,
    search_type=search_type,
)

unrouted_judge = run_judge(
    question=comparison_question3,
    rag_output=unrouted_out,
    search_type=search_type,
)

print_comparison_result(
    title="WITHOUT QUESTION ROUTING",
    question_type=None,
    output=unrouted_out,
    judge=unrouted_judge,
)


WITHOUT QUESTION ROUTING
Question type: None — generic baseline

--- Answer ---
1. **Summary:**
- The documented clinical history includes breast cancer with breast biopsy and HER2 testing, and the only active condition listed is malignant neoplasm of overlapping sites of the left breast, with adjustment disorder also documented as resolved.
- The most recent clinically important status in the overview is that cancer disease progression was recorded as improved on 2022-05-20, and treatment status changed on that same date.
- Oncology timeline shows breast cancer workup in June 2021 with biopsy and HER2 testing followed by chemotherapy on 2021-06-25 and 2021-07-16.

2. **Active Conditions:**
- **Malignant neoplasm of overlapping sites of left female breast** — active; date: 2021-06-15
- **Adjustment disorder with mixed emotional features** — resolved; date: 2021-06-15

3. **Medications:**
- No current medication is documented in the provided medication snapshot.

4. **Oncology timeline

## With question routing: 

Router selects among 4 question types (Patient overview, Conditions, Medications, Oncology Timeline) and controls the prompt and retrieval (which now covers selected category-specific document types and document headings).

Comparison question 1:
The router selects `prompt_mode="extract_conditions"`, which uses CONDITIONS_EXTRA to do structured condition extraction.
The judge's result is the same as without routing (relevance_score is 2, groundedness_score: 1, overall_score: 1.5, RELEVANT, PARTLY_GROUNDED), but now the answer correctly lists all the active diagnoses and separates medical diagnoses from socio-economic findings.

In [18]:
from question_router import route_question

route = route_question(comparison_question)

print("=== Router output ===")
print(f"Suggested question type: {route.question_type}")
print(f"Confidence: {route.confidence:.0%}")
print(json.dumps(route.scores, indent=2))


routed_out = rag.rag_new(
    query=comparison_question,
    patient_id=patient_id,
    question_type=route.question_type,
    num_results=num_results,
    model=model,
    search_type=search_type,
)

routed_judge = run_judge(
    question=comparison_question,
    rag_output=routed_out,
    search_type=search_type,
)

print_comparison_result(
    title="WITH QUESTION ROUTING",
    question_type=route.question_type,
    output=routed_out,
    judge=routed_judge,
)

=== Router output ===
Suggested question type: conditions
Confidence: 100%
{
  "patient_overview": 0.0,
  "conditions": 5.0,
  "medications": 0.0,
  "oncology_timeline": 0.0
}

WITH QUESTION ROUTING
Question type: conditions

--- Answer ---
**Diagnoses and disorders**
- Hypoxemia (disorder) — active; date: 2020-05-04.
- Prediabetes — active; date: 2019-01-29.
- Malignant neoplasm of breast (disorder) — active; date: 2015-05-07.
- Body mass index 30+ - obesity (finding) — active; date: 2002-01-15.
- Osteoarthritis of knee — active; date: 2004-07-12.
- Viral sinusitis (disorder) — resolved; date: 2021-02-06.
- Acute pulmonary embolism (disorder) — resolved; date: 2020-05-11.
- Pneumonia (disorder) — resolved; date: 2020-05-04.
- Sepsis caused by virus (disorder) — resolved; date: 2020-05-04.
- Concussion with loss of consciousness — resolved; date: 2022-04-22.
- Sprain of wrist — resolved; date: 2012-09-08.
- Injury of medial collateral ligament of knee — resolved; date: 1979-03-02.

**F

In [29]:
from question_router import route_question

route = route_question(comparison_question3)

print("=== Router output ===")
print(f"Suggested question type: {route.question_type}")
print(f"Confidence: {route.confidence:.0%}")
print(json.dumps(route.scores, indent=2))


routed_out = rag.rag_new(
    query=comparison_question3,
    patient_id=patient_id,
    question_type=route.question_type,
    num_results=num_results,
    model=model,
    search_type=search_type,
)

routed_judge = run_judge(
    question=comparison_question3,
    rag_output=routed_out,
    search_type=search_type,
)

print_comparison_result(
    title="WITH QUESTION ROUTING",
    question_type=route.question_type,
    output=routed_out,
    judge=routed_judge,
)

=== Router output ===
Suggested question type: patient_overview
Confidence: 80%
{
  "patient_overview": 4.0,
  "conditions": 0.0,
  "medications": 0.0,
  "oncology_timeline": 0.0
}

WITH QUESTION ROUTING
Question type: patient_overview

--- Answer ---
1. **Summary:**
- The documented clinical conditions include malignant neoplasm of the breast with stage IIIA/stage 3 disease, with HER2 negative status and T3/N1/M0 findings in the oncology timeline.
- The most recent clinically important status shows the cancer disease progression as improved and cancer treatment status changed on 2022-05-20, with completed chemotherapy procedures around that period.
- Oncology timeline synopsis: Breast cancer was diagnosed as active on 2021-06-14 with stage group clinical cancer stage 3A/stage 3 documented on 2021-06-15, followed by repeated completed chemotherapy procedures from 2021-06-25 through 2022-05-14 and an improved disease progression observation on 2021-12-06 and 2022-05-20.

2. **Active Con

# A low complexity patient

In [21]:
patient_id = "d65197b3-056a-2136-b584-77f43c29da3f"

## Without question routing: `question_type=None`


In [24]:
unrouted_out = rag.rag_new(
    query=comparison_question2,
    patient_id=patient_id,
    question_type=None,
    num_results=num_results,
    model=model,
    search_type=search_type,
)

unrouted_judge = run_judge(
    question=comparison_question2,
    rag_output=unrouted_out,
    search_type=search_type,
)

print_comparison_result(
    title="WITHOUT QUESTION ROUTING",
    question_type=None,
    output=unrouted_out,
    judge=unrouted_judge,
)


WITHOUT QUESTION ROUTING
Question type: None — generic baseline

--- Answer ---
1. **Summary:**
- The record documents breast cancer treatment with epirubicin hydrochloride injections and tamoxifen, and later ribociclib on the oncology medication timeline.
- The latest medication entries are completed orders rather than active ongoing prescriptions, with the most recent dated 2022-05-20.
- Oncology timeline shows repeated epirubicin injections from 2021-12-17 through 2022-05-14 and tamoxifen/ribociclib on 2022-05-20, all documented as completed.

2. **Active Conditions:**
No qualifying clinical conditions documented.

3. **Medications:**
- **Active analgesic regimen** — none documented.
- Epirubicin hydrochloride injection
- Tamoxifen
- Ribociclib

4. **Oncology timeline:**
No oncology timeline information documented.

--- Judge evaluation ---
{
  "relevance_score": 2,
  "groundedness_score": 1,
  "overall_score": 1.5,
  "relevance_label": "RELEVANT",
  "groundedness_label": "PARTLY_G

## With question routing: 


In [26]:
from question_router import route_question

route = route_question(comparison_question2)

print("=== Router output ===")
print(f"Suggested question type: {route.question_type}")
print(f"Confidence: {route.confidence:.0%}")
print(json.dumps(route.scores, indent=2))


routed_out = rag.rag_new(
    query=comparison_question2,
    patient_id=patient_id,
    question_type=route.question_type,
    num_results=num_results,
    model=model,
    search_type=search_type,
)

routed_judge = run_judge(
    question=comparison_question2,
    rag_output=routed_out,
    search_type=search_type,
)

print_comparison_result(
    title="WITH QUESTION ROUTING",
    question_type=route.question_type,
    output=routed_out,
    judge=routed_judge,
)

=== Router output ===
Suggested question type: medications
Confidence: 100%
{
  "patient_overview": 0.0,
  "conditions": 0.0,
  "medications": 6.0,
  "oncology_timeline": 0.0
}

WITH QUESTION ROUTING
Question type: medications

--- Answer ---
No current medication is documented in the provided medication snapshot.

- **Tamoxifen 10 MG Oral Tablet** — completed; date: 2022-05-20; documented in the patient_overview.md medications section.
- **ribociclib 200 MG Oral Tablet** — completed; date: 2022-05-20; documented in the patient_overview.md medications section.
- **Epirubicin Hydrochloride 2 MG/ML Injection** — completed; date: 2022-05-14; documented repeatedly in the patient_overview.md medications section (most recent date retained).

--- Judge evaluation ---
{
  "relevance_score": 2,
  "groundedness_score": 1,
  "overall_score": 1.5,
  "relevance_label": "RELEVANT",
  "groundedness_label": "PARTLY_GROUNDED",
  "explanation": "The answer addresses the question by listing medications f